# exp_6 · 02 detector — train, tau on val, quality on test, coverage (BAR-ATH)
Streams the tactical video (sequential decode passes), trains the 3-frame U-Net on train blocks, infers on
hidden + val + test frames, links, picks tau on val, measures quality once on test and coverage on hidden
graded frames. Resumable (per-pass checkpoints, inference chunks). **Reads BAR-ATH video and ATD: aggregates
only; never open this notebook executed.** Results: `results/detector_quality.json`.

In [1]:
from pathlib import Path
import os, sys, json, time, subprocess
from datetime import datetime, timezone
ROOT = Path.cwd()
if not (ROOT / 'exp6_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_6')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import exp6_common as C
import exp1_common as C1
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
def now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')
def make_log(name):
    path = C.LOG_DIR / f'{name}_progress.log'
    path.parent.mkdir(parents=True, exist_ok=True)
    def log(message, **_):
        print(message, flush=True)
        with path.open('a') as handle:
            handle.write(f'{now()} {message}\n')
    return log
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])

GUARD_FILE = 'guard_02.json'
guard = C.guard_check(C.RESULTS / 'write_guard_before.json', C.RESULTS / GUARD_FILE)
print('nothing written under exp_0-exp_5 / data since freezing:', guard['all_unchanged'],
      {k: v['unchanged'] for k, v in guard['folders'].items()})
if not guard['all_unchanged']:
    raise RuntimeError('Write guard changed: diagnose before continuing')

import exp6_recover as R
log = make_log('02')

PREREG verified dca69fd7188c7ef8 | frozen 2026-11-10T14:57:58+00:00


nothing written under exp_0-exp_5 / data since freezing: True {'exp_0': True, 'exp_1': True, 'exp_2': True, 'exp_3': True, 'exp_4': True, 'exp_5': True, 'data': True, 'data_metric_repo': True}


In [2]:
summary = R.recover(C.CACHE, log)
C.write_results(C.RESULTS / 'detector_quality.json', {**summary, 'computed_at_utc': now()})
print('tau:', summary['tau'])
print('val:', summary['val'])
print('test:', summary['test'])
print('coverage of hidden graded frames:', summary['coverage'], '| runs:', summary['run_lengths'])
print('bridge:', summary['bridge'])

step 500/10496 mean loss 0.26020 elapsed 114s


step 1000/10496 mean loss 0.09817 elapsed 237s


step 1500/10496 mean loss 0.10776 elapsed 373s


step 2000/10496 mean loss 0.09457 elapsed 479s


step 2500/10496 mean loss 0.08784 elapsed 603s


pass 1/4 done: 167901 samples, 629s


... (36 outputs left out)

inference 90000/102764 frames, 5983s


inference 95000/102764 frames, 6319s


inference 100000/102764 frames, 6635s


tau: 0.9
val: {'frames': 19169, 'accepted': 6779, 'inliers': 4427, 'precision': 0.6530461720017702, 'recall': 0.230945797902864, 'sigma_m': 0.4759418511554467, 'fp_rate': 0.3469538279982298, 'median_error_inliers_m': 0.4177324700591643}
test: {'frames': 18952, 'accepted': 5742, 'inliers': 4366, 'precision': 0.7603622431208638, 'recall': 0.23037146475306036, 'sigma_m': 0.48320129491003044, 'fp_rate': 0.23963775687913624, 'median_error_inliers_m': 0.4310545981956465}
coverage of hidden graded frames: 0.31160373126247237 | runs: {'runs': 671, 'frames': 20143, 'mean': 30.019374068554395, 'median': 3.0, 'p10': 1.0, 'p90': 44.0, 'max': 2978, 'share_frames_in_runs_ge_25': 0.8641711760909497}
bridge: {'gap_start': {'n': 32, 'median_m': 0.8516824099683955, 'p90_m': 57.20780646707067, 'share_le_2m': 0.59375, 'share_le_5m': 0.6875}, 'gap_end': {'n': 22, 'median_m': 2.7498298725484958, 'p90_m': 46.461566615047786, 'share_le_2m': 0.5, 'share_le_5m': 0.5454545454545454}, 'both': {'n': 54, 'median_m'